# <img align="left" src="./图片/movie_camera.png"     style=" width:40px;  " > 实践实验：协同过滤推荐系统

在本练习中，你将实现协同过滤，为电影构建一个推荐系统。

# <img align="left" src="./图片/film_reel.png"     style=" width:40px;  " > 目录
- [ 1 - 记号](#1)
- [ 2 - 推荐系统](#2)
- [ 3 - 电影评分数据集](#3)
- [ 4 - 协同过滤学习算法](#4)
  - [ 4.1 协同过滤代价函数](#4.1)
    - [ 练习 1](#ex01)
- [ 5 - 学习电影推荐](#5)
- [ 6 - 推荐结果](#6)
- [ 7 - 恭喜！](#7)




##  程序包 <img align="left" src="./图片/film_strip_vertical.png"     style=" width:40px;   " >
我们将使用大家已经熟悉的 NumPy 和 Tensorflow 程序包。

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 记号


| 一般 记号 | 说明 | Python（如有） |
|:---|:---|:---|
| $r(i,j)$ | 标量；若用户 j 给游戏 i 打了分则为 1，否则为 0 |  |
| $y(i,j)$ | 标量；用户 j 给游戏 i 的评分（在 r(i,j) = 1 有定义时） |  |
| $\mathbf{w}^{(j)}$ | 向量；用户 j 的参数 |  |
| $b^{(j)}$ | 标量；用户 j 的参数 |  |
| $\mathbf{x}^{(i)}$ | 向量；电影 i 的特征评分 |  |
| $n_u$ | 用户数量 | num_users |
| $n_m$ | 电影数量 | num_movies |
| $n$ | 特征数量 | num_features |
| $\mathbf{X}$ | 向量 $\mathbf{x}^{(i)}$ 构成的矩阵 | X |
| $\mathbf{W}$ | 向量 $\mathbf{w}^{(j)}$ 构成的矩阵 | W |
| $\mathbf{b}$ | 偏置参数 $b^{(j)}$ 构成的向量 | b |
| $\mathbf{R}$ | 元素 $r(i,j)$ 构成的矩阵 | R |



<a name="2"></a>
## 2 - 推荐系统 <img align="left" src="./图片/film_rating.png" style=" width:40px;  " >
在本实验中，你将实现协同过滤学习算法，并将其应用于一个电影评分数据集。
协同过滤推荐系统的目标是生成两组向量：对每个用户，生成一个体现其电影品味的“参数向量”；对每部电影，生成一个同样长度的特征向量，体现对这部电影的某种描述。两组向量的点积再加上偏置项，应给出该用户可能给这部电影打分的估计值。

下图详细说明了这些向量是如何学习得到的。

<figure>
   <img src="./图片/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >

> **图中文字（中文）**：b(0)、b(1)、w(0)、w(1)；用户 1……；电影/条目 1、2、3 学习参数 一般形式：…… w(j)、x(i)、b(j)
</figure>

已有的评分以矩阵形式给出，如下所示。$Y$ 包含评分：0.5 到 5（含端点），步长 0.5；未评分的电影为 0。$R$ 在电影已被评分处为 1。电影按行排列，用户按列排列。每个用户有一个参数向量 $w^{user}$ 和偏置。每部电影有一个特征向量 $x^{movie}$。这些向量以已有的用户/电影评分作为训练数据同时学习得到。上面展示了一个训练样本：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量 $x^{movie}$ 必须同时满足所有用户，而用户向量 $w^{user}$ 必须同时满足所有电影。这正是该方法名称的由来——所有用户协同（collaborate）产生评分集合。

<figure>
   <img src="./图片/ColabFilterUse.PNG"  style="width:640px;height:250px;" >

> **图中文字（中文）**：用户 b(0) b(1) w(0) w(1)…… 电影/条目 1、2、3 用参数预测未评分电影的评分 w(1)、x(0)、b(1) → 预测评分
</figure>

一旦特征向量和参数学习完成，就可以用它们预测用户可能给未评分电影打多少分。上图展示了这一点。该方程是预测用户一对电影零评分的一个示例。


在本练习中，你将实现函数 `cofiCostFunc`，用于计算协同过滤的目标函数。实现目标函数之后，你将用 TensorFlow 自定义训练循环来学习协同过滤的参数。第一步是详细说明本实验将使用的数据集和数据结构。

<a name="3"></a>
## 3 - 电影评分数据集 <img align="left" src="./图片/film_rating.png"     style=" width:40px;  " >
该数据集源自 [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) 数据集。
[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含 600 位用户对 9000 部电影的评分。为聚焦 2000 年以后的电影，数据集已做过缩减。该数据集的评分范围为 0.5 到 5，步长 0.5。缩减后的数据集有 $n_u = 443$ 位用户、$n_m= 4778$ 部电影。

下面，你将把电影数据集加载到变量 $Y$ 和 $R$ 中。

矩阵 $Y$（$n_m \times n_u$ 矩阵）存储评分 $y^{(i,j)}$。矩阵 $R$ 是二值指示矩阵，若用户 $j$ 给电影 $i$ 打过分，则 $R(i,j) = 1$，否则 $R(i,j)=0$。

在练习的这一部分中，你还将使用矩阵 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$：

$$\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

$\mathbf{X}$ 的第 $i$ 行对应第 $i$ 部电影的特征向量 $x^{(i)}$，$\mathbf{W}$ 的第 $j$ 行对应第 $j$ 位用户的一个参数向量 $\mathbf{w}^{(j)}$。$x^{(i)}$ 和 $\mathbf{w}^{(j)}$ 都是 $n$ 维向量。就本练习而言，你将使用 $n=10$，因此 $\mathbf{x}^{(i)}$ 和 $\mathbf{w}^{(j)}$ 各有 10 个元素。相应地，$\mathbf{X}$ 是 $n_m \times 10$ 矩阵，$\mathbf{W}$ 是 $n_u \times 10$ 矩阵。

我们将先加载电影评分数据集，以理解数据的结构。
我们将用电影数据集加载 $Y$ 和 $R$。
我们还会用预先算好的值加载 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。这些值将在实验后面学习得到，但我们会先用预先算好的值来搭建代价模型。

In [ ]:
#加载数据
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

In [ ]:
#  从矩阵中，我们可以计算出平均评分等统计量。
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

<a name="4"></a>
## 4 - 协同过滤学习算法 <img align="left" src="./图片/film_filter.png"     style=" width:40px;  " >

现在，你开始实现协同过滤学习算法。你将先实现目标函数。

在电影推荐场景中，协同过滤算法考虑一组 $n$ 维参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，其中模型把用户 $j$ 对电影 $i$ 的评分预测为 $y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(i)}$。给定一个由若干用户对若干电影给出的评分组成的数据集，你希望学习出拟合最好（使平方误差最小）的参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$。

你将补全 cofiCostFunc 中的代码，计算协同过滤的代价函数。


<a name="4.1"></a>
### 4.1 协同过滤代价函数

协同过滤的代价函数由下式给出
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\underbrace{
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
}_{regularization}
\tag{1}$$
(1) 中的第一个求和表示“对所有满足 $r(i,j)$ 等于 $1$ 的 $i$、$j$”，也可以写成：

$$
= \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\text{regularization}
$$

你现在应编写 cofiCostFunc（协同过滤代价函数）使其返回该代价值。

<a name="ex01"></a>
### 练习 1

**用 for 循环实现：**
先用以 for 循环的方式实现代价函数。
建议分两步开发代价函数。第一步，先实现不带正则化的代价函数。下面提供了一个不含正则化的测试用例来检验你的实现。跑通之后，再加入正则化，并运行包含正则化的测试。注意，只有当 $R(i,j) = 1$ 时，你才应累加用户 $j$ 和电影 $i$ 的代价值。

In [ ]:
# 计分函数：cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """返回基于内容过滤的代价
    参数：
      X (ndarray (num_movies,num_features)): 物品特征矩阵
      W (ndarray (num_users,num_features)) : 用户参数矩阵
      b (ndarray (1, num_users)            : 用户参数向量
      Y (ndarray (num_movies,num_users)    : 用户对电影评分的矩阵
      R (ndarray (num_movies,num_users)    : 矩阵，其中若第 i 部电影被第 j 个用户评过分，则 R(i, j) = 1
      lambda_ (float): 正则化参数
    返回：
      J (float) : 代价"""
    nm, nu = Y.shape
    J = 0
    ### 在此处编写代码 ###
    
        
        
        
            
            
            
            
    
    
    ### 代码编写结束 ###

    return J

In [ ]:
# 公开测试
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    你可以仿照 (1) 中的求和，用两个 for 循环来组织代码。
    先实现不带正则化的代码。
    注意 (1) 中有些元素是向量。请使用 np.dot()。你也可以使用 np.square()。
    特别注意哪些元素以 i 为索引、哪些以 j 为索引。别忘了除以二。
    

```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```
    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>
        
    这里还有更多细节。下面的代码在使用矩阵元素前先把它们逐个取出。
    也可以直接引用矩阵。
    这段代码不含正则化。
    

```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```

    
<details>
    <summary><font size="2" color="darkblue"><b>最后的办法（完整的不含正则化实现）</b></font></summary>
    

```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```

    
<details>
    <summary><font size="2" color="darkblue"><b>正则化</b></font></summary>
     正则化就是把 W 数组和 X 数组的每个元素平方，再把所有平方项求和。
     你可以使用 np.square() 和 np.sum()。

<details>
    <summary><font size="2" color="darkblue"><b>正则化细节</b></font></summary>
    

```python 
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
```

    
</details>
</details>
</details>
</details>

    


In [ ]:
# 缩小数据集规模，使运行更快
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# 计算代价函数
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

**预期输出（lambda = 0）**：
$13.67$。

In [ ]:
# 计算带正则化的代价函数
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

**预期输出**：

28.09

**向量化实现**

创建一个向量化实现来计算 $J$ 很重要，因为后面优化过程中会多次调用它。这里用到的线性代数不是本系列的重点，因此实现已直接给出。如果你是线性代数专家，欢迎不参考下面的代码自行编写。

运行下面的代码，确认它产生的结果与未向量化的版本相同。

In [ ]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """返回基于内容过滤的代价
    为提速做了向量化。使用 tensorflow 运算，以兼容自定义训练循环。
    参数：
      X (ndarray (num_movies,num_features)): 物品特征矩阵
      W (ndarray (num_users,num_features)) : 用户参数矩阵
      b (ndarray (1, num_users)            : 用户参数向量
      Y (ndarray (num_movies,num_users)    : 用户对电影评分的矩阵
      R (ndarray (num_movies,num_users)    : 矩阵，其中若第 i 部电影被第 j 个用户评过分，则 R(i, j) = 1
      lambda_ (float): 正则化参数
    返回：
      J (float) : 代价"""
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [ ]:
# 计算代价函数
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# 计算带正则化的代价函数
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

**预期输出**：
Cost: 13.67
Cost (with regularization): 28.09

<a name="5"></a>
## 5 - 学习电影推荐 <img align="left" src="./图片/film_man_action.png" style=" width:40px;  " >
------------------------------

完成协同过滤代价函数的实现后，你就可以开始训练算法，为自己生成电影推荐了。

在下面的单元格中，你可以填入自己看过的电影。算法随后会为你给出推荐！我们已按自己的偏好填了一些值，但在我们的选择跑通之后，你应把它改成符合你自己口味的内容。
数据集中所有电影的清单见文件 [电影列表](数据/small_movie_list.csv)。

In [ ]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  初始化我的评分

# 查看 small_movie_list.csv 文件以了解我们数据集中每部电影的 id
# 例如，玩具总动员 3（Toy Story 3, 2010）的 ID 是 2700，要给它打 "5" 分，可以设置
my_ratings[2700] = 5 

#或者假设你不喜欢劝导（Persuasion, 2007），可以设置
my_ratings[2609] = 2;

# 我们挑选了几部我们喜欢/不喜欢的电影，以及我们给出的评分
# 给出的评分如下：
my_ratings[929]  = 5   # 指环王：王者归来（The Lord of the Rings: The Return of the King）
my_ratings[246]  = 5   # 怪物史莱克（Shrek, 2001）
my_ratings[2716] = 3   # 盗梦空间（Inception）
my_ratings[1150] = 5   # 超人总动员（The Incredibles, 2004）
my_ratings[382]  = 2   # 天使爱美丽（Amelie）
my_ratings[366]  = 5   # 哈利·波特与魔法石（Harry Potter and the Sorcerer's Stone, 2001）
my_ratings[622]  = 5   # 哈利·波特与密室（Harry Potter and the Chamber of Secrets, 2002）
my_ratings[988]  = 3   # 美丽心灵的永恒阳光（Eternal Sunshine of the Spotless Mind, 2004）
my_ratings[2925] = 1   # 路易· Theroux：法律与无序（Louis Theroux: Law & Disorder, 2008）
my_ratings[2937] = 1   # 无物申报（Nothing to Declare / Rien à déclarer）
my_ratings[793]  = 5   # 加勒比海盗：黑珍珠号的诅咒（Pirates of the Caribbean: The Curse of the Black Pearl, 2003）
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');

现在，让我们把这些评分加入 $Y$ 和 $R$，并对评分做归一化。

In [ ]:
# 重新加载评分并加入新的评分
Y, R = load_ratings_small()
Y    = np.c_[my_ratings, Y]
R    = np.c_[(my_ratings != 0).astype(int), R]

# 对数据集做归一化
Ynorm, Ymean = normalizeRatings(Y, R)

让我们准备训练模型。初始化参数并选择 Adam 优化器。

In [ ]:
#  有用的取值
num_movies, num_users = Y.shape
num_features = 100

# 设置初始参数 (W, X)，用 tf.Variable 跟踪这些变量
tf.random.set_seed(1234) # 为得到一致的结果
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# 实例化一个优化器。
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

现在让我们训练协同过滤模型。这将学习出参数 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。

同时学习 $w$、$b$ 和 $x$ 所涉及的操作，不属于 TensorFlow 神经网络包提供的典型“层（layer）”。因此，课程 2 中使用的流程（Model、Compile()、Fit()、Predict()）在这里并不直接适用。我们可以改用自定义训练循环。

回忆前面实验中梯度下降的步骤。
- 重复直到收敛：
    - 计算前向传播
    - 计算损失相对于参数的导数
    - 用学习率和算出的导数更新参数
    
TensorFlow 有为你计算导数的强大能力。如下所示。在 `tf.GradientTape()` 作用域内，对 Tensorflow 变量的操作会被记录。之后调用 `tape.gradient()` 时，它会返回损失相对于被记录变量的梯度。随后可以用优化器把梯度应用于参数。
这是对 TensorFlow 及其他机器学习框架这一实用特性的非常简短的介绍。更多信息可在你感兴趣的框架中查阅“自定义训练循环（custom training loops）”。
    


In [ ]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # 使用 TensorFlow 的 GradientTape
    # 记录用于计算代价的运算
    with tf.GradientTape() as tape:

        # 计算代价（前向传播已包含在代价中）
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # 用梯度带自动获取
    # 可训练变量关于损失的梯度
    grads = tape.gradient( cost_value, [X,W,b] )

    # 通过更新来运行一步梯度下降
    # 使损失最小的变量取值。
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # 定期记录日志。
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

<a name="6"></a>
## 6 - 推荐结果
下面，我们为所有电影和用户计算评分，并显示推荐出来的电影。这些推荐基于上面以 `my_ratings[]` 形式录入的电影和评分。要预测用户 $j$ 对电影 $i$ 的评分，你计算 $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$。用矩阵乘法即可为所有评分算出该值。

In [ ]:
# 用训练好的权重和偏置做一次预测
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

#恢复均值
pm = p + Ymean

my_predictions = pm[:,0]

# 对预测结果排序
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

实践中，还可以利用额外信息来增强我们的预测。上面，前几百部电影的预测评分都落在一个较小的区间内。我们可以在上述结果基础上进一步筛选：从这些靠前的电影中选出平均评分高的电影，以及评分人数超过 20 的电影。本节使用 [Pandas](https://pandas.pydata.org/) 数据框，它有许多方便的排序功能。

In [ ]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

<a name="7"></a>
## 7 - 恭喜！ <img align="left" src="./图片/film_award.png"     style=" width:40px;  " >
你已经实现了一个很有用的推荐系统！